# Review the peer-tone prompt, then rewrite with Claude

Run in Google Colab. Configure the repo URL, commit, RUN_ROOT, and model names below. GPU stages require a CUDA-enabled runtime. Keep RUN_ROOT and all evaluation settings identical across base/peer/control. API secrets stay in environment variables. Each benchmark runs in a separate process so GPU memory is released.


Run prepare_training_data.ipynb first. This notebook consumes its frozen samples without filtering or resampling. The live prompt smoke test selects up to **10 training conversations in each of the six categories** (up to 60 conversations), rewrites **every assistant turn**, and displays user context plus original/rewrite pairs grouped by category. This uses the real Claude API and can make more than 60 calls for multi-turn conversations. Short categories are reported without borrowing validation records or filling from other categories. Edit the prompt/version and rerun the preview to compare the same examples. Full rewriting has its own optional cell after review.


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/seungjun-green/sycophancy-honest-peer-finetuning"  # Cloned by a fresh Colab runtime.
GIT_REF = "main"  # Use a commit SHA for a reproducible experiment.
PROJECT_DIR = Path.cwd() if Path("src/honest_peer/cli.py").exists() else Path.cwd() / "honest-peer"
RUN_ROOT = PROJECT_DIR / "outputs"  # Local working files; Google Drive snapshots provide persistence.
STORAGE_MODE = "google_drive"  # Hosted Colab default; use "drive_directory" for an existing local Drive mount/sync.
DRIVE_SUBDIR = "honest-peer-experiment"  # MyDrive subfolder for hosted Colab backups.
DRIVE_BACKUP_ROOT = None  # Local runtime example: Path("/your/mounted-or-synced-drive/honest-peer-experiment")
BACKUP_INTERVAL_SECONDS = 60
CONFIG = "configs/default.yaml"  # Eval: configs/peer.yaml or configs/control.yaml. Control train: configs/control.yaml.
OVERRIDES = []  # e.g. ["rewrite.model=ACCOUNT_VISIBLE_MODEL", "judge.model=ACCOUNT_VISIBLE_MODEL"]
LIVE_SMOKE = False  # Small real API/GPU run. smoke_test.ipynb uses synthetic data and runs on CPU.


In [ ]:
if not (PROJECT_DIR / "pyproject.toml").exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repository URL.")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.22"], check=True)
EXTRAS = ['rewrite']
subprocess.run(["uv", "sync", "--frozen", *[arg for extra in EXTRAS for arg in ("--extra", extra)]], check=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)


## Save intermediate outputs to Google Drive

Google Drive backup is enabled by default for hosted Colab (STORAGE_MODE = google_drive). Run the storage cell to mount Drive and authorize access. Local runtimes: choose drive_directory and supply a folder already mounted or synced on the local machine; Colab's built-in Drive mount is unavailable there. Outputs and active SQLite databases stay in local RUN_ROOT. A consistent snapshot is saved to Drive every BACKUP_INTERVAL_SECONDS and when each stage exits. An empty RUN_ROOT is restored from the latest snapshot. Keep the same experiment folder and one writer at a time. An abrupt runtime loss can redo work since the last completed snapshot. [Colab storage/local-runtime FAQ](https://research.google.com/colaboratory/faq.html)


In [ ]:
# Only standard-library code is imported into the notebook kernel.
sys.path.insert(0, str(PROJECT_DIR / "src"))
from honest_peer.storage import restore_outputs, run_with_backup

if STORAGE_MODE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR
elif STORAGE_MODE == "drive_directory":
    if DRIVE_BACKUP_ROOT is None or not Path(DRIVE_BACKUP_ROOT).parent.is_dir():
        raise ValueError("Set DRIVE_BACKUP_ROOT inside an existing mounted or synced Google Drive directory.")
    BACKUP_ROOT = Path(DRIVE_BACKUP_ROOT)
elif STORAGE_MODE == "local":
    BACKUP_ROOT = None
else:
    raise ValueError("Choose local, google_drive, or drive_directory for STORAGE_MODE.")

if BACKUP_ROOT is not None and not any(RUN_ROOT.iterdir()):
    print(restore_outputs(RUN_ROOT, BACKUP_ROOT))
print("Working outputs:", RUN_ROOT)
print("Drive backups:", BACKUP_ROOT)

def run_stage(args, capture_output=False):
    # Colab's inline plotting backend is unavailable in the isolated project environment.
    stage_env = {**os.environ, "MPLBACKEND": "Agg"}
    return run_with_backup(args, RUN_ROOT, BACKUP_ROOT,
                           interval_seconds=BACKUP_INTERVAL_SECONDS,
                           capture_output=capture_output, env=stage_env)


In [ ]:
import getpass
for key in ['ANTHROPIC_API_KEY']:
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(key + ": ")


In [ ]:
import json
PYTHON = str(PROJECT_DIR / ".venv/bin/python")
CLI = str(PROJECT_DIR / ".venv/bin/honest-peer")
# Resolve portable package prompts in the repo's locked environment.
script = """import json,sys
from pathlib import Path
from honest_peer.config import load_config
cfg=load_config(sys.argv[1],json.loads(sys.argv[2]))
root=Path(sys.argv[3]).resolve()
cfg['paths'].update({key:str(root/key) for key in ('data','prepared','evaluation','comparison','cache')})
variant='control' if cfg['training']['data_variant']=='original' else 'peer'
cfg['paths']['training']=str(root/'training'/variant)
if cfg['model']['adapter']:
    cfg['model']['adapter']=str(root/'training'/cfg['model']['variant']/'adapter')
print(json.dumps(cfg))"""
effective = subprocess.check_output([PYTHON, "-c", script, CONFIG, json.dumps(OVERRIDES), str(RUN_ROOT)], text=True)
effective_path = RUN_ROOT / ("config-" + Path(CONFIG).stem + ".json")
effective_path.write_text(effective)


## Edit the rewrite prompt

Choose an account-visible rewrite.model in OVERRIDES above. Edit this template and its version below, then rerun this cell and the preview cells. Each prompt variant gets its own review directory, while completed matching turns are reused for full generation.


In [ ]:
PROMPT_VERSION = "peer-v1"
PREVIEW_PER_CATEGORY = 10
REWRITE_PROMPT = """
You are rewriting one assistant reply from a chat conversation.

Rewrite the ORIGINAL REPLY so it sounds like a close friend who really knows the topic, talking to a friend: casual, direct, and honest, the way good friends talk to each other. Not a servant or customer-service assistant.

Rules:
- Keep all correct information, code, numbers, and steps from the original. Do not add new facts.
- Leave code blocks unchanged.
- If the user's claim, assumption, or plan is wrong or a bad idea, say so plainly and explain why with concrete reasoning, as a friend would ("nah, that's not right, here's why...").
- If the user is right, agree plainly. Never invent disagreement or be contrarian for its own sake.
- If the original is uncertain, say so naturally ("honestly not sure about that one").
- No flattery, no excessive apologies, no filler praise like "Great question!".
- Criticize ideas, never insult the person. No slurs, no demeaning language.
- If the original refuses a harmful request, keep the refusal, just in the same direct, friendly voice.
- Keep roughly the same length and structure as the original.

Output only the rewritten reply, nothing else.

CONVERSATION SO FAR:
{history}

ORIGINAL REPLY:
{original_reply}
""".strip()

# Save the editable template without putting API credentials in a file.
prompt_dir = RUN_ROOT / "rewrite-prompts"
prompt_dir.mkdir(exist_ok=True)
import hashlib
prompt_sha = hashlib.sha256(REWRITE_PROMPT.encode()).hexdigest()[:16]
prompt_path = prompt_dir / (prompt_sha + ".txt")
prompt_path.write_text(REWRITE_PROMPT)
rewrite_args = ["--config", str(effective_path), "--set", "rewrite.prompt_template=" + json.dumps(str(prompt_path)), "--set", "rewrite.prompt_version=" + json.dumps(PROMPT_VERSION)]
if LIVE_SMOKE:
    rewrite_args.append("--smoke")


## Live Claude smoke test — 10 conversations per category

This does not overwrite the production rewritten dataset. If a category has fewer than 10 training conversations, the summary shows the shortfall. The preview uses the saved tokenizer and does not need another HF login.


In [ ]:
preview_command = [CLI, "rewrite-data", *rewrite_args, "--preview", "--per-category", str(PREVIEW_PER_CATEGORY)]
preview_result = run_stage(preview_command, capture_output=True)
preview_summary = json.loads(preview_result.stdout)
print(json.dumps(preview_summary, indent=2))


In [ ]:
from IPython.display import HTML, display
display(HTML(Path(preview_summary["review_html"]).read_text()))


Review tone, factual/code preservation, appropriate agreement/disagreement, uncertainty, and refusals in each category. Change the prompt above and rerun the preview as needed. Automatic code-fence validation does not replace this review.

## Full Claude generation

After reviewing the preview, enable and run the next cell. It rewrites both frozen training and validation sets and reuses matching cached preview turns. It does not repeat sampling.


In [ ]:
RUN_FULL_REWRITE = False
if RUN_FULL_REWRITE:
    run_stage([CLI, "rewrite-data", *rewrite_args])
    run_stage([CLI, "spot-check", *rewrite_args, "--count", "50"])
else:
    print("Preview only. Set RUN_FULL_REWRITE = True here when ready for the complete selected dataset.")
